# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rashidsami10000-afk/rashid-flyrank-internship-ml-owncopy/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook frames the question my capstone will answer. The lane below is **provisional** — I can confirm or change it until the end of Week 4, so this is a commitment to a direction, not a wall. Every number shown here was produced by running the code cells in this notebook against the 30,000-row anonymized starter dataset that ships in the repo.

# 1) My lane (or freestyle) and why

**I pick Lane 4 — CTR / Engagement Opportunity Scoring.**

The lane guide phrase this lane answers:

> Which visible pages under-capture clicks or engagement and deserve metadata, content, or monitoring review?

My version of that question, narrowed to a decision support tool:

> **Given a page's position tier, which pages sit far *below* the CTR that position should earn — and is content_type (keyword / feedly / comparison article) part of the gap?**

Why this lane and not the others:

- **Lane 1 (signal analysis)** is descriptive — it tells you what correlates with clicks. Lane 4 pushes one step further into a *ranked list an analyst can act on*, which matches the course's final output: ranked action recommendations.
- **Lane 2 (refresh scoring)** is about *which pages to refresh*. Many of those pages aren't refresh candidates at all — they just under-capture clicks. Lane 4 keeps the queue honest by separating "traffic problem" from "engagement problem."
- **Lane 3 (clustering)** is interesting but its archetypes are a lens, not an action. Lane 4 ends in an explicit per-page action.

What attracted me: the starter data shows a large, same-tier CTR gap by content type (shown below). If that survives the volume checks and the bigger warehouse, it means the queue could be re-ranked cheaply by a transparent, explainable signal.


In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Volume floor so small low-volume pages don't dominate a tier's mean.
visible = df[df["impressions_90d"] >= 100]

# Mean CTR by content_type, split by position tier (ctr is a x100 percentage: 0.35 = 0.35%).
pivot = visible.groupby(["position_tier", "content_type"])["ctr"].mean().unstack(fill_value=0)
pivot = pivot.reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
print("Mean CTR (%) by position_tier x content_type")
print(pivot.round(4).to_string())

print("\nRows per content_type in the page_1 tier:")
page1 = visible[visible["position_tier"] == "page_1"]
print(page1.groupby("content_type").size().to_string())

# Volume caveat for the extreme tiers - never read a tier mean without its volume floor.
print("\nMedian impressions_90d by position_tier (the 'how loud is this number' check):")
print(df.groupby("position_tier")["impressions_90d"].median().round(0).to_string())


Mean CTR (%) by position_tier x content_type
content_type   comparison article  feedly article  keyword article
position_tier                                                     
top_3                      0.0000          2.9000           0.3104
page_1                     0.1412          0.9048           0.3458
striking                   0.1474          0.3580           0.2559
page_3_5                   0.0940          0.1656           0.1427
deep                       0.0000          0.0440           0.0555

Rows per content_type in the page_1 tier:
content_type
comparison article     226
feedly article         221
keyword article       8186

Median impressions_90d by position_tier (the 'how loud is this number' check):
position_tier
deep         218.0
page_1      1180.0
page_3_5     812.0
striking     874.0
top_3          3.0


# 2) The question: decision, action, and the cost of a wrong call

**What decision does this improve?**
How FlyRank's refresh/review queue is ordered. Today the queue leans on the decline label and ad-hoc thresholds; this work replaces "feels urgent" with "sits well below its position's expected CTR" as a structured, inspectable reason to review a page **now**.

**Who acts on the output, and what exactly do they do?**
A FlyRank content analyst opens the ranked queue, reads the score + reason code, and takes one of the lane-guided actions: rewrite the title or meta description, improve intent match, improve snippet/section structure, improve on-page engagement, or decide the page is fine and **monitor**. The output is not an automatic edit — it decides **which pages the human reviews first**.

**What does a wrong answer cost?**
- A **false positive** (a page flagged as under-capturing that isn't) burns the analyst's scarcest resource — attention — on a page that won't improve, delaying real fixes.
- A **false negative** (a page that genuinely under-captures never surfaces) hides a fixable click leak from the client.

Because attention is the scarce resource, the metric for the queue is **precision@K** — of the top K pages the analyst actually reviews, how many really under-capture for their position. I won't optimise accuracy on all 30k pages; I'll optimise the top of the list.


In [2]:
import pandas as pd
import json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 2: demand keywords barely explain impressions -> page-level analysis is needed.
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"corr(search_volume, impressions_90d) = {corr:.3f}")

# Number 3: on the starter pipeline a learned ranking beats the hand-rule baseline at the top of the queue.
res = json.load(open("outputs/model_results.json", encoding="utf-8"))
baseline_p50 = res["baseline"]["baseline_precision_at_50"]
rf_p50 = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline (hand rule) Precision@50  = {baseline_p50:.2f}")
print(f"Random Forest Precision@50        = {rf_p50:.2f}")
print(f"Lift at the top of the queue      = {rf_p50 / baseline_p50:.1f}x")


corr(search_volume, impressions_90d) = 0.001
Baseline (hand rule) Precision@50  = 0.24
Random Forest Precision@50        = 0.74
Lift at the top of the queue      = 3.1x


# 3) Quick look at the data — the numbers that make this lane worth 7 weeks

Three real numbers from the starter dataset, each produced by the code cells above:

1. **Same-tier CTR gap by content type.** In the `page_1` tier (mean position ≤ 10, the rows that most matter to a reviewer), mean CTR is **0.90% for feedly articles, 0.35% for keyword articles, 0.14% for comparison articles** — a **6.4× spread between the best- and worst-performing formats at the same position** (feedly n=221, keyword n=8,186, comparison n=226). If that is real and not confounded, content_type is a cheap, transparent re-ranking signal.

2. **Demand keywords don't explain what a page actually got.** `corr(search_volume, impressions_90d) = 0.001` — the target keyword's search volume says almost nothing about the page's realized 90-day impressions. So you cannot rank the queue by "big keyword up front"; you have to look at realized performance per page.

3. **A learned ranking beat the transparent baseline at exactly the point the queue is used.** On the starter pipeline (client-holdout split), Random Forest reached **Precision@50 = 0.74** vs the hand-rule baseline's **0.24** — a **~3.1× lift in the share of the top-50 that were right**. This is evidence (on a 30k-row slice) that a model can find useful structure the manual thresholds miss.

**Honest volume caveats**, because the lane guide insists:
- The `top_3` and `deep` tier means are small-n (top_3 feedly n=5, comparison n=1) and low-volume — median impressions in `top_3` is only **3 per 90 days**, where one click moves CTR by ~30 percentage points. I exclude those tiers from any same-position claim and set a minimum-volume floor (e.g. impressions_90d ≥ 100 plus a minimum click count) before trusting a CTR.
- A gap at the mean is not a per-page truth; it's a *starting hypothesis* for the audit weeks.


# 4) Careful words: what I can and can't claim

**What I can say (all observed / measured / directional / decision-support):**

- *Observed:* in the starter slice, pages in the `page_1` tier had different mean CTR by content_type (0.90% / 0.35% / 0.14%). This is a fact about the slice, not an algorithm law.
- *Observed:* search_volume and impressions_90d are almost uncorrelated (r ≈ 0.001) in this slice.
- *Measured:* on the starter client-holdout pipeline, RF Precision@50 (0.74) beat the hand-rule baseline (0.24).
- These numbers support *which pages an analyst should look at first* — they do not prove what will happen after the edit.

**What I can't claim:**

- I cannot claim content_type *causes* the CTR difference. The starter slice is observational; the formats may differ in topic mix, word count, or intent — all confounders I'll need to control in the audit.
- I cannot claim CTR at `top_3`/`deep` "proves" anything — n and volume are too small there; claims need a volume floor.
- I cannot claim my queue predicts a page's future recovery. Lane 4 ranks *review priority*, not *outcome after editing*. Recovery would be a different (Lane 2/freestyle) project with its own leakage audit.
- I cannot generalise from 32 pseudonymized clients / 30k rows to all of FlyRank without re-earning the result on the ~79M-row warehouse.
- I will not use `trend_direction` / `trend_pct` as features (they are the label's source), and I won't treat the decline label as "CTR truth" — this lane is about clicks vs position, a separate quantity.


# 5) Self-check

- [x] I picked **one of the four predefined lanes** — Lane 4 (CTR / Engagement Opportunity Scoring) — with a precise opening question.
- [x] I named the **decision** (queue ordering / who gets the review slot), the **actor** (content analyst) and the **action** (review first: rewrite title/meta, improve intent match, or monitor).
- [x] I named the **cost of a wrong call** (lost analyst attention on false positives; hidden click leaks on false negatives) and chose **precision@K** as the matching metric.
- [x] I showed **3 real numbers from the starter data** — the page_1 CTR gap by content_type, the ~0 search_volume↔impressions correlation, and the Precision@50 lift — each computed in a code cell above.
- [x] I explained why this is **not just "train a model"** — it's decision support for a human reviewer, with a volume floor, confounder checks, and a ranked queue as the unit of value.
- [x] My claims use **careful words**: observed / measured / directional / decision-support. No causal or Google-algorithm claims.
- [x] **Public-safe:** no client names, URLs, private queries, or raw text anywhere in this repo.
- [x] The notebook **executes top to bottom** (Runtime → Run all) and the outputs are saved in this committed version.


In [3]:
import pandas as pd
import json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
visible = df[df["impressions_90d"] >= 100]
page1 = visible[visible["position_tier"] == "page_1"]
ctr = page1.groupby("content_type")["ctr"].mean().round(4)

print("page_1 tier mean CTR (%) by content_type:")
print(ctr.sort_values(ascending=False).to_string())
print(f"Widest same-tier spread: {ctr.max() / ctr.min():.1f}x")
print()

res = json.load(open("outputs/model_results.json", encoding="utf-8"))
print(f"RF Precision@50:  {res['models']['random_forest']['precision_at_50']:.2f}")
print(f"Baseline P@50:    {res['baseline']['baseline_precision_at_50']:.2f}")
print("\nAll numbers re-checked against the committed starter dataset and outputs. Notebook ready.")


page_1 tier mean CTR (%) by content_type:


content_type
feedly article        0.9048
keyword article       0.3458
comparison article    0.1412
Widest same-tier spread: 6.4x

RF Precision@50:  0.74
Baseline P@50:    0.24

All numbers re-checked against the committed starter dataset and outputs. Notebook ready.
